# Giggsdance — MiniMax H3 at 60 fps

You are inside Modal, so **you are already authenticated. There is no token to paste.**

Set the notebook's own hardware to **CPU** (the toolbar above). The heavy work happens in separate Modal functions on a B200 — a GPU attached to *this* notebook would sit idle and still be billed.

### Before you run anything: check your credits

Look at the banner at the top of the Modal dashboard. **If it says you have about $1 of free credits, stop and read the "Budget reality" cell at the bottom first.** Self-hosting H3 means downloading ~90 GB of weights and loading ~124 GB into a GPU before a single frame exists, and that does not fit in $1.

### Licence

The [MiniMax H3 licence](https://github.com/Hvkki/minimax/blob/main/NOTICE.md) grants **no rights in the EU, UK, South Korea or USA**, and the restriction covers the model's **outputs**, not only its weights. Mark anything you publish as AI-generated.

## 1. Get the code

`!` runs a shell command. Without it the cell is parsed as Python and raises `SyntaxError`.

In [ ]:
!git clone -q https://github.com/Hvkki/minimax.git /root/minimax
%cd /root/minimax
!pip install -q pytest
print("ready")

## 2. Free check — costs nothing

Validates the interpolation, geometry and encoding stages on synthetic frames: frame counts, fps, bit depth, colour tags, A/V sync. No GPU, no weights, no charge.

In [ ]:
!python run.py --dry-run

## 3. Cheap check — steps 1 to 3 only, no GPU

Local environment, weight download, and the unit suite run **inside** the container. Skips the expensive GPU probe.

**This is where the ~90 GB weight download happens** (on cheap CPU). It is a one-time cost and later runs skip it.

In [ ]:
!modal run doctor.py --skip-gpu

## 4. Full check — boots a B200 and loads H3

Loads ~124 GB and compares our generation call against the pipeline's real signature. This is the one thing that de-risks `stages/generate.py`, the only module in the repo its author could never execute.

Costs roughly the model-load time at $6.25/hr. **Skip this if you are on $1 of credits.**

In [ ]:
!modal run doctor.py

## 5. Render

Defaults are the cheap ones: 5 s, `native` (no super-resolution — it measured at ~84% of post-processing time), 60 fps, 8 steps. `--budget-usd` becomes a hard container timeout, so an overrun is killed rather than billed.

In [ ]:
!modal run run.py --budget-usd 1.0

## 6. Watch it

In [ ]:
from pathlib import Path
from IPython.display import Video, display

found = sorted(Path("/root/minimax").glob("*.mp4"))
display(Video(str(found[-1]), embed=True, width=720)) if found else print("nothing rendered yet")

## Budget reality

Modal's free tier is **$30/month, but only ~$1 is available until a payment method is added**. Self-hosting H3 does not fit in $1, and it is better to know that now:

| | Cost driver |
|---|---|
| Weight download | ~90 GB on CPU, one time |
| Volume storage | ~90 GB held for as long as you keep it |
| Model load | ~124 GB into GPU memory, **before any frame is generated**, every cold start |
| Render | the actual generation, on top of all of the above |

The model load alone can consume most of a dollar. With **$30** unlocked, everything in this notebook is comfortable — a 5-second clip at native resolution is a small fraction of it.

**If you only have $1 and do not want to add a card**, self-hosting is the wrong tool. Use MiniMax's own hosted API instead: it is globally available (no territory restriction), needs no weights, no 124 GB load and no storage, and you pay per clip. You also get `H3-Context-IR` and native 2K via `H3-Regenerate-2K`, neither of which is open source. This repo's prompt builder, 60 fps conversion, upscaling and encoding all still apply on top of API output.

## Troubleshooting

| Symptom | Cause |
|---|---|
| `Bad Request: Unsupported URL` on import | you pasted the **repo** URL; Import from URL wants a link to a `.ipynb` file |
| `SyntaxError: invalid syntax` | shell command in a Python cell — add `!` |
| `ModuleNotFoundError: giggsdance` | wrong directory — `%cd /root/minimax` |
| timeout | raise `--budget-usd`, or lower `--steps` |
| generation fails | `!modal run run.py --describe` prints the real pipeline signature |